# Analyse de logs & utilisation de pandas

Ce notebook analyse le journal d'accès d'un serveur web nginx : un échantillon (environ une ligne sur cent) du journal d'une boutique en ligne iranienne, du 22 au 26 janvier 2019, publié par Farzin Zaker sur Harvard Dataverse sous licence CC0. Chaque ligne décrit une requête, au format `combined` de nginx (https://nginx.org/en/docs/http/ngx_http_log_module.html#log_format), suivi ici d'un dernier champ :

```
66.249.66.194 - - [22/Jan/2019:03:57:11 +0330] "GET /filter/b36,p3 HTTP/1.1" 200 35145 "-" "Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)" "-"
```

On y lit l'adresse IP du client, deux champs vides (`-`), la date et l'heure (à Téhéran, `+0330`), la requête (méthode, chemin, protocole), le code de statut de la réponse, sa taille en octets, la page d'où vient le client (le référent, `-` s'il n'y en a pas), son navigateur (*user agent*), et l'en-tête `X-Forwarded-For`, toujours vide ici.

1. Découper une ligne avec une expression régulière
2. Charger le journal dans une `DataFrame`
3. Analyser le trafic : codes de statut, chemins les plus demandés, trafic par heure, requêtes suspectes
4. Une commande pour filtrer le journal par date

Les exercices 1 et 2 s'enchaînent ; les analyses du 3 utilisent la `DataFrame` du 2 et sont indépendantes entre elles ; le 4 ne dépend d'aucun autre. Chaque exercice a une cellule pour votre code, puis des conseils supplémentaires et sa solution, repliés : ouvrez-les si besoin. Les solutions définissent ce dont la suite a besoin : si vous sautez un exercice, exécutez sa solution.

Les deux cellules suivantes téléchargent le journal, puis le lisent dans la liste `lignes`.

In [ ]:
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/nginx-log/access-small.log

In [ ]:
import re
from pathlib import Path

import pandas as pd

lignes = Path("access-small.log").read_text(encoding="utf8").splitlines()
print(f"{len(lignes)} lignes")
print(lignes[3])

## 1. Découper une ligne avec une expression régulière

Écrivez un motif `MOTIF`, compilé avec `re.compile`, dont les groupes nommés découpent une ligne du journal en ses champs : `ip`, `date`, `methode`, `chemin`, `protocole`, `statut`, `taille`, `referent` et `agent` (le dernier champ, toujours vide, n'a pas besoin de groupe). Le flag `re.VERBOSE` permet de l'écrire sur plusieurs lignes, commentées.

Écrivez ensuite une fonction `decouper(ligne)` qui renvoie le dictionnaire des groupes du motif, ou `None` si la ligne ne lui correspond pas. Affichez le découpage de la première ligne, puis le nombre de lignes du journal qui ne correspondent pas au motif, et les 100 premiers caractères de chacune : qu'ont-elles de particulier ?

Pour mettre au point le motif, https://regex101.com/ (saveur « Python ») explique chacune de ses parties et montre ce qu'il reconnaît dans une ligne que vous y collez.

In [ ]:
# Votre code ici


def decouper(ligne: str) -> dict[str, str] | None: ...

### Conseils supplémentaires

Un groupe nommé `(?P<nom>...)` par champ, avec ce que le champ peut contenir :

- `\S+` reconnaît une suite de caractères qui ne sont pas des blancs : l'adresse IP, la méthode, le chemin
- `[^\]]+` reconnaît tout jusqu'au crochet fermant (la date), `[^"]*` tout jusqu'au guillemet suivant (le protocole, le référent, le navigateur) ; nginx écrit `\x22` un guillemet qui fait partie d'un champ
- `\d{3}` et `\d+` reconnaissent le code de statut et la taille

Avec `re.VERBOSE`, les blancs du motif sont ignorés et `#` commence un commentaire : une espace à reconnaître s'écrit `\ `, de préférence en début de ligne, où un éditeur ne l'efface pas. `MOTIF.match(ligne)` renvoie `None` si le début de la ligne ne correspond pas au motif, sinon un objet `Match`, dont `groupdict()` donne le dictionnaire des groupes.

Vous devez trouver, pour la première ligne, `'ip': '54.36.149.41'`, `'date': '22/Jan/2019:03:56:14 +0330'`, `'methode': 'GET'`, `'statut': '200'` et `'taille': '30577'`. Deux lignes ne correspondent pas : une requête si longue que nginx l'a tronquée et refusée (code 414), et des octets qui ne sont pas du HTTP (le début d'une connexion chiffrée, envoyé au port HTTP, que nginx refuse avec un code 400).

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
MOTIF = re.compile(
  r"""
  (?P<ip>\S+)\ \S+\ \S+  # Adresse IP, puis deux champs vides (-)
  \ \[(?P<date>...)\]  # Date, entre crochets
  \ "(?P<methode>...)\ (?P<chemin>...)\ (?P<protocole>...)"  # Requête
  \ (?P<statut>...)\ (?P<taille>...)  # Code de statut, taille
  \ "(?P<referent>...)"\ "(?P<agent>...)"  # Référent, navigateur
  """,
  re.VERBOSE,
)


def decouper(ligne: str) -> dict[str, str] | None:
  correspondance = MOTIF.match(ligne)
  if ...:
    return None
  return ...


print(decouper(lignes[0]))
rejetees = [ligne for ligne in lignes if ...]
print(f"{len(rejetees)} lignes rejetées :")
for ligne in rejetees:
  print(...)
```

### Solution

In [ ]:
MOTIF = re.compile(
  r"""
  (?P<ip>\S+)\ \S+\ \S+  # Adresse IP, puis deux champs vides (-)
  \ \[(?P<date>[^\]]+)\]  # Date, entre crochets
  \ "(?P<methode>\S+)\ (?P<chemin>\S+)\ (?P<protocole>[^"]*)"  # Requête
  \ (?P<statut>\d{3})\ (?P<taille>\d+)  # Code de statut, taille
  \ "(?P<referent>[^"]*)"\ "(?P<agent>[^"]*)"  # Référent, navigateur
  """,
  re.VERBOSE,
)


def decouper(ligne: str) -> dict[str, str] | None:
  correspondance = MOTIF.match(ligne)
  if correspondance is None:
    return None
  return correspondance.groupdict()


print(decouper(lignes[0]))
rejetees = [ligne for ligne in lignes if decouper(ligne) is None]
print(f"{len(rejetees)} lignes rejetées :")
for ligne in rejetees:
  print(ligne[:100])

## 2. Charger le journal dans une `DataFrame`

Construisez la `DataFrame` `logs` : une ligne par ligne du journal que `decouper` découpe, une colonne par groupe du motif. Toutes ses colonnes sont alors des chaînes de caractères : convertissez `date` en dates avec `pd.to_datetime`, puis `statut` et `taille` en entiers. Affichez les types des colonnes, la période que couvre le journal (sa première et sa dernière date) et les premières lignes de `logs`.

**Documentation utile**

- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [Codes de format de `strptime`](https://docs.python.org/fr/3/library/datetime.html#format-codes)
- [`Series.astype`](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`pd.DataFrame(dictionnaires)` crée, à partir d'une liste de dictionnaires, une colonne par clef. Gardez les lignes découpées seulement : `decouper` renvoie `None` pour les autres. `pd.to_datetime(colonne, format=...)` lit les dates au format donné, avec les codes de `strptime` : `%d` le jour, `%b` le mois abrégé en anglais (`Jan`), `%Y` l'année, `%H:%M:%S` l'heure, `%z` le décalage horaire (`+0330`). `colonne.astype(int)` convertit une colonne de chaînes en entiers. `min()` et `max()` donnent la plus petite et la plus grande valeur d'une colonne.

Vous devez trouver le type `datetime64[ns, UTC+03:30]` pour `date`, `int64` pour `statut` et `taille`, et un journal du 22 janvier 2019 à 3 h 56 au 26 janvier à 20 h 29.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
decoupees = [... for ligne in lignes]
logs = pd.DataFrame([d for d in decoupees if ...])
logs["date"] = pd.to_datetime(logs["date"], format="...")
logs["statut"] = ...
logs["taille"] = ...
print(logs.dtypes)
print(f"Du {logs['date'].min()} au {...}")
logs.head()
```

### Solution

In [ ]:
decoupees = [decouper(ligne) for ligne in lignes]
logs = pd.DataFrame([d for d in decoupees if d is not None])
logs["date"] = pd.to_datetime(logs["date"], format="%d/%b/%Y:%H:%M:%S %z")
logs["statut"] = logs["statut"].astype(int)
logs["taille"] = logs["taille"].astype(int)
print(logs.dtypes)
print(f"Du {logs['date'].min()} au {logs['date'].max()}")
logs.head()

## 3. Analyser le trafic

Quatre analyses de `logs`, indépendantes entre elles.

### 3.1. Codes de statut

Comptez les réponses de chaque code de statut. Quelle part des requêtes a échoué, c'est-à-dire reçu un code d'au moins 400 (4xx : erreur du client, 5xx : erreur du serveur) ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`value_counts()` compte chaque valeur d'une colonne, de la plus fréquente à la moins fréquente. La comparaison `logs["statut"] >= 400` donne une `Series` de booléens : sa somme compte les `True`, et sa moyenne en donne la proportion. Dans une f-string, le format `:.2%` affiche une proportion en pourcentage, avec deux décimales.

Vous devez trouver 95 912 réponses 200 (succès), 3 379 réponses 304 (non modifié : le navigateur garde la copie qu'il a déjà), et 1 717 erreurs, soit 1,66 % des requêtes.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(logs["statut"].value_counts())
erreurs = ...
print(f"{erreurs.sum()} erreurs, soit {...:.2%} des requêtes")
```

#### Solution

In [ ]:
print(logs["statut"].value_counts())
erreurs = logs["statut"] >= 400
print(f"{erreurs.sum()} erreurs, soit {erreurs.mean():.2%} des requêtes")

### 3.2. Chemins les plus demandés

Affichez les 10 chemins les plus demandés : ce sont surtout des fichiers que chaque page charge, comme des images ou des polices. Affichez ensuite les 10 chemins les plus demandés parmi les réponses 404 (introuvable) : ce sont les fichiers à créer, ou les liens à corriger.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`value_counts()` trie déjà par fréquence décroissante : `head(10)` en garde les 10 premiers. Un masque booléen, `logs["statut"] == 404`, dans `logs.loc[masque, "chemin"]`, ne garde que les chemins des réponses 404.

Vous devez trouver `/settings/logo` en tête (3 673 requêtes) et, parmi les 404, des icônes que les iPhone et les iPad demandent quand on ajoute un site à l'écran d'accueil : `/apple-touch-icon-precomposed.png` (179 requêtes), `/apple-touch-icon.png`…

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(logs["chemin"].value_counts().head(10))
introuvables = logs.loc[..., "chemin"]
print(...)
```

#### Solution

In [ ]:
print(logs["chemin"].value_counts().head(10))
introuvables = logs.loc[logs["statut"] == 404, "chemin"]
print(introuvables.value_counts().head(10))

### 3.3. Trafic par heure

Comptez les requêtes de chaque heure de la journée (de 0 à 23, à l'heure du serveur), toutes journées confondues, et tracez-les en diagramme en bâtons. Calculez aussi le volume transféré à chaque heure, en mégaoctets : la somme des tailles des réponses. À quelle heure le site est-il le plus chargé ?

**Documentation utile**

- [`Series.dt.hour`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.hour.html)
- [`Series.sort_index`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_index.html)
- [`Series.plot.bar`](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.bar.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`logs["date"].dt.hour` donne l'heure de chaque date. Son `value_counts()` compte les requêtes de chaque heure, mais les trie par fréquence : `sort_index()` remet les heures dans l'ordre. `.plot.bar()` trace une `Series` en bâtons, un par valeur de son index ; ses arguments `xlabel` et `ylabel` nomment les axes. `logs.groupby(heures)["taille"].sum()` additionne les tailles des réponses de chaque heure : divisez par `1e6` pour des mégaoctets.

Vous devez trouver un creux à 5 h (707 requêtes, 13,5 Mo), un pic de requêtes à 11 h (7 315) et un pic de volume à 13 h et 14 h (86,9 Mo chacune) : le trafic reste haut de 10 h à 15 h.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
heures = logs["date"].dt.hour
requetes = ...
requetes.plot.bar(xlabel="Heure", ylabel="Requêtes")
volume = logs.groupby(heures)["taille"].sum() / ...
print(volume.round(1))
```

#### Solution

In [ ]:
heures = logs["date"].dt.hour
requetes = heures.value_counts().sort_index()
requetes.plot.bar(xlabel="Heure", ylabel="Requêtes")
volume = logs.groupby(heures)["taille"].sum() / 1e6
print(volume.round(1))

### 3.4. Requêtes suspectes

La boutique tourne sur Grails, un framework Java : elle n'a aucune page PHP. Une requête dont le chemin contient `.php` cherche donc une faille connue ailleurs : une interface d'administration de base de données (phpMyAdmin), la page de connexion de WordPress, un script laissé par un pirate sur un autre site… Trouvez ces requêtes, et affichez leur adresse IP, leur date, leur méthode, leur chemin et leur code de statut. Comptez ensuite les requêtes de chaque adresse IP, et de chaque code de statut : le serveur a-t-il exécuté l'une d'elles ?

**Documentation utile**

- [`Series.str.contains`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`logs["chemin"].str.contains(".php", regex=False)` dit si chaque chemin contient `.php` : sans `regex=False`, le texte cherché serait une expression régulière, où le point reconnaît n'importe quel caractère. `logs.loc[masque, colonnes]` garde les lignes du masque et les colonnes de la liste `colonnes`.

Vous devez trouver 17 requêtes, venues de 12 adresses IP. Aucune n'a reçu de code 200 : 14 ont été redirigées (301), une n'a pas trouvé son fichier (404), et les deux qui tentent de faire télécharger un programme au serveur avec `wget` ont été refusées (400).

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
php = logs["chemin"].str.contains(...)
suspectes = logs.loc[php, [...]]
print(suspectes["ip"].value_counts())
print(...)
suspectes
```

#### Solution

In [ ]:
php = logs["chemin"].str.contains(".php", regex=False)
suspectes = logs.loc[php, ["ip", "date", "methode", "chemin", "statut"]]
print(suspectes["ip"].value_counts())
print(suspectes["statut"].value_counts())
suspectes

## 4. Une commande pour filtrer le journal par date

Un administrateur veut extraire d'un journal les lignes de quelques jours, sans ouvrir de notebook. Écrivez le script `filtre_logs.py` : la cellule de l'exercice l'écrit dans un fichier, grâce à la commande magique `%%writefile` de sa première ligne. Avec `argparse`, il prend :

- un argument positionnel `journal`, le chemin du fichier à filtrer ;
- deux options facultatives, `--debut` et `--fin`, des dates au format `AAAA-MM-JJ`, toutes deux incluses.

Il affiche, telles quelles, les lignes du journal dont la date est dans cet intervalle : le shell les redirige dans un fichier avec `>`. Une date mal écrite, ou un début après la fin, est une erreur que signale `argparse`. Le script n'a pas besoin de pandas : il lit le journal ligne à ligne, ce qui marche aussi pour un fichier de plusieurs gigaoctets.

Exécutez la cellule de l'exercice pour écrire le script, puis la suivante pour le lancer : elle affiche son aide, extrait les lignes du 24 janvier 2019 dans `jour.log`, compte les lignes de ce fichier et affiche les deux dernières.

**Documentation utile**

- [`%%writefile`](https://ipython.readthedocs.io/en/stable/interactive/magics.html#cellmagic-writefile)
- [`date.fromisoformat`](https://docs.python.org/fr/3/library/datetime.html#datetime.date.fromisoformat)
- [`datetime.strptime`](https://docs.python.org/fr/3/library/datetime.html#datetime.datetime.strptime)
- [`ArgumentParser.error`](https://docs.python.org/fr/3/library/argparse.html#argparse.ArgumentParser.error)

In [ ]:
%%writefile filtre_logs.py
# Votre code ici

In [ ]:
!python filtre_logs.py --help
!python filtre_logs.py access-small.log --debut 2019-01-24 --fin 2019-01-24 > jour.log
!wc -l jour.log
!tail -n 2 jour.log

### Conseils supplémentaires

`add_argument("--debut", type=date.fromisoformat)` convertit l'option en `date`, et `argparse` signale une date que `fromisoformat` refuse ; une option absente vaut `None`. De même, `type=Path` fait de l'argument `journal` un `Path`, que sa méthode `open` ouvre. `parser.error(message)` affiche l'usage du script et le message, puis l'arrête. Pour la date d'une ligne, un motif plus simple que celui de l'exercice 1 suffit : `\[([^\]]+)\]` capture le texte entre les premiers crochets, que `datetime.strptime(texte, format)` lit avec le format de l'exercice 2, et dont `.date()` garde le jour. `print(ligne, end="")` affiche une ligne sans ajouter de saut de ligne : elle a déjà le sien.

Vous devez trouver 18 579 lignes le 24 janvier 2019. `--debut 2019-01-25 --fin 2019-01-24` et `--debut 2019-01-32` doivent afficher une erreur.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
%%writefile filtre_logs.py
import argparse
import re
from datetime import date, datetime
from pathlib import Path

DATE = re.compile(r"\[([^\]]+)\]")


def main() -> None:
  parser = argparse.ArgumentParser(description="...")
  parser.add_argument("journal", type=Path, help="...")
  parser.add_argument("--debut", type=..., help="...")
  parser.add_argument("--fin", ...)
  args = parser.parse_args()
  if ...:
    parser.error("...")
  with args.journal.open(encoding="utf8") as journal:
    for ligne in journal:
      correspondance = DATE.search(ligne)
      if correspondance is None:
        continue
      jour = datetime.strptime(..., "%d/%b/%Y:%H:%M:%S %z").date()
      if (args.debut is None or ...) and (...):
        print(ligne, end="")


if __name__ == "__main__":
  main()
```

### Solution

In [ ]:
%%writefile filtre_logs.py
import argparse
import re
from datetime import date, datetime
from pathlib import Path

DATE = re.compile(r"\[([^\]]+)\]")


def main() -> None:
  parser = argparse.ArgumentParser(description="Filtre un journal nginx par date.")
  parser.add_argument("journal", type=Path, help="Chemin du journal à filtrer.")
  parser.add_argument(
    "--debut", type=date.fromisoformat, help="Première date gardée (AAAA-MM-JJ)."
  )
  parser.add_argument(
    "--fin", type=date.fromisoformat, help="Dernière date gardée (AAAA-MM-JJ)."
  )
  args = parser.parse_args()
  if args.debut is not None and args.fin is not None and args.debut > args.fin:
    parser.error("--debut est après --fin")
  with args.journal.open(encoding="utf8") as journal:
    for ligne in journal:
      correspondance = DATE.search(ligne)
      if correspondance is None:
        continue
      jour = datetime.strptime(correspondance[1], "%d/%b/%Y:%H:%M:%S %z").date()
      if (args.debut is None or args.debut <= jour) and (
        args.fin is None or jour <= args.fin
      ):
        print(ligne, end="")


if __name__ == "__main__":
  main()

In [ ]:
!python filtre_logs.py --help
!python filtre_logs.py access-small.log --debut 2019-01-24 --fin 2019-01-24 > jour.log
!wc -l jour.log
!tail -n 2 jour.log
!python filtre_logs.py access-small.log --debut 2019-01-25 --fin 2019-01-24
!python filtre_logs.py access-small.log --debut 2019-01-32